# SVM Model

Train and evaluate a Support Vector Machine classifier.

In [1]:
print('SVM notebook placeholder')

SVM notebook placeholder


In [9]:
import os
import cv2
import numpy as np
import joblib
from pathlib import Path
from skimage.feature import hog

dataset_dir = "./dataset_20_species"
OUTPUT_DIR = Path("./processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not os.path.exists("./processed_data/X_hog.npy"):
    print("Feature vectors not found. Generating X_hog.npy now...")

    if not os.path.exists(dataset_dir):
        import tarfile
        tar_path = "CUB_200_2011.tgz"
        extract_dir = "./raw_cub"
        dataset_url = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz"

        if not os.path.exists(tar_path):
            !wget --no-check-certificate {dataset_url} -O {tar_path}

        if not os.path.exists(extract_dir):
            with tarfile.open(tar_path, "r:gz") as tar:
                tar.extractall(path=extract_dir)

        source_images_path = os.path.join(extract_dir, "CUB_200_2011", "images")
        os.makedirs(dataset_dir, exist_ok=True)
        all_species = sorted(os.listdir(source_images_path))
        bd_keywords = ['Crow', 'Kingfisher', 'Hummingbird', 'Mallard', 'Warbler',
                       'Towhee', 'Jay', 'Creeper', 'Waxwing', 'Cuckoo',
                       'Thrush', 'Woodpecker', 'Wren', 'Vireo', 'Catbird',
                       'Meadowlark', 'Blackbird', 'Gull', 'Tern', 'Pelican']
        selected = []
        for s in all_species:
            if any(kw.lower() in s.lower() for kw in bd_keywords):
                if s not in selected: selected.append(s)
            if len(selected) == 20: break

        import shutil
        for s in selected:
            shutil.copytree(os.path.join(source_images_path, s), os.path.join(dataset_dir, s))

    X_hog, y_labels = [], []
    classes = sorted([d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))])
    label_mapping = {c: i for i, c in enumerate(classes)}

    for c in classes:
        c_path = os.path.join(dataset_dir, c)
        for img_name in os.listdir(c_path):
            if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                img = cv2.imread(os.path.join(c_path, img_name), cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    img_resized = cv2.resize(img, (128, 128))
                    feat = hog(img_resized, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), block_norm='L2-Hys')
                    X_hog.append(feat)
                    y_labels.append(label_mapping[c])

    np.save(OUTPUT_DIR / "X_hog.npy", np.array(X_hog, dtype=np.float32))
    np.save(OUTPUT_DIR / "y_labels.npy", np.array(y_labels, dtype=np.int64))
    joblib.dump(label_mapping, OUTPUT_DIR / "label_mapping.pkl")
    print("HOG feature vectors generated and saved to ./processed_data!")
else:
    print("Processed feature files already exist. Ready to train!")

Processed feature files already exist. Ready to train!


##Setup & Load Saved HOG Features

In [10]:
import os
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

DATA_DIR = "./processed_data"

X_hog = np.load(os.path.join(DATA_DIR, "X_hog.npy"))
y_labels = np.load(os.path.join(DATA_DIR, "y_labels.npy"))
label_mapping = joblib.load(os.path.join(DATA_DIR, "label_mapping.pkl"))

print(f"Loaded HOG Feature Matrix X: {X_hog.shape}")
print(f"Loaded Target Labels y: {y_labels.shape}")
print(f"Number of classes: {len(label_mapping)}")

Loaded HOG Feature Matrix X: (1168, 8100)
Loaded Target Labels y: (1168,)
Number of classes: 20


##Stratified Train-Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_hog,
    y_labels,
    test_size=0.20,
    random_state=42,
    stratify=y_labels
)

print(f"Training Samples: {X_train.shape[0]}")
print(f"Testing Samples : {X_test.shape[0]}")

Training Samples: 934
Testing Samples : 234


##Train Baseline SVM Classifier

In [12]:
print("--- Training Baseline SVM Classifier ---")

svm_baseline = SVC(kernel='rbf', C=1.0, random_state=42)
svm_baseline.fit(X_train, y_train)


y_pred_base = svm_baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred_base)

print(f"\nBaseline SVM Test Accuracy: {baseline_acc * 100:.2f}%")

--- Training Baseline SVM Classifier ---

Baseline SVM Test Accuracy: 20.51%
